# Урок 18. Контрольная работа № 2

11 класс · III четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [← Урок 17](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-17.ipynb) · [Урок 19 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-19.ipynb)

---

Разбор ошибок. Самостоятельная работа: графы и базы данных.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 11А
Класс = "" #@param {type:"string"}
#@markdown
#@markdown Адрес журнала даёт учитель. Пусто — в конце урока
#@markdown получишь квитанцию, её нужно будет отправить ему сам.
Журнал = "" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="11-18", name=ФИО, klass=Класс,
         journal=Журнал)

## Где обычно ошибаются

Четверть была про связи: графы, деревья и базы данных — три способа
описать, что с чем связано. Разберём типичные ошибки и напишем
контрольную.

### Ошибка 1. Обход без отметки «уже были»

В графе с циклом обход без множества посещённых зациклится намертво.
Отметка ставится **в момент постановки в очередь**, а не когда
вершину достали, — иначе одна вершина попадёт в очередь несколько раз.

### Ошибка 2. BFS применили к взвешенному графу

Обход в ширину находит путь с наименьшим **числом рёбер**, а не
с наименьшей суммой весов. Как только у рёбер появились веса, нужен
Дейкстра.

### Ошибка 3. Ноль в весовой матрице приняли за дорогу

Ноль обычно означает «дороги нет». Проверка `if веса[i][j] > 0`
обязательна, иначе появятся маршруты через несуществующие связи.

### Ошибка 4. У рекурсии по дереву нет базового случая для листа

У листа список потомков пуст, и цикл не выполняется — но если
функция возвращает что-то только внутри цикла, для листа вернётся
`None`. Пишите явный `return` для листа.

### Ошибка 5. Первичный ключ выбран по «говорящему» полю

Фамилия, телефон, название — всё это меняется и повторяется.
Первичный ключ должен быть уникальным, непустым и неизменным.

### Ошибка 6. Условие на группу написано в WHERE

`WHERE` работает до группировки и с агрегатными функциями
не сочетается. Условие вида «групп больше двух» — только в `HAVING`.

### Ошибка 7. COUNT(*) при LEFT JOIN

`COUNT(*)` считает строки, а строка есть даже там, где пары
не нашлось, — и вместо честного нуля получается единица.
Считайте `COUNT(поле_второй_таблицы)`.

### Чек-лист перед сдачей

1. В обходе есть множество посещённых?
2. Для взвешенного графа взят Дейкстра, а не BFS?
3. У рекурсии по дереву есть возврат для листа?
4. В запросе с группировкой поля в SELECT только из GROUP BY
   и агрегаты?
5. Для «у кого ничего нет» использован LEFT JOIN?

## Разбираем сложное

### Пример 1. Граф и дерево — одна и та же рекурсия

Дерево — частный случай графа, и функции для него получаются
из графовых выбрасыванием проверки на повторное посещение.
Сравним подсчёт вершин в обоих случаях.

In [ ]:
дерево = {"А": ["Б", "В"], "Б": ["Г"], "В": [], "Г": []}
граф = {"А": ["Б", "В"], "Б": ["А", "Г"], "В": ["А"], "Г": ["Б"]}


def вершин_дерева(дерево, корень):
    итог = 1
    for потомок in дерево[корень]:
        итог += вершин_дерева(дерево, потомок)
    return итог


def вершин_графа(граф, старт, посещённые=None):
    if посещённые is None:
        посещённые = set()
    посещённые.add(старт)
    for сосед in граф[старт]:
        if сосед not in посещённые:
            вершин_графа(граф, сосед, посещённые)
    return len(посещённые)


print("В дереве:", вершин_дерева(дерево, "А"))
print("В графе: ", вершин_графа(граф, "А"))

Различие ровно в одной строке — проверке `if сосед not in посещённые`.
Уберите её из графовой версии, и функция уйдёт в бесконечную
рекурсию: А → Б → А → Б.

### Пример 2. Один вопрос — SQL и Python

«Сколько книг у каждого автора» решается и запросом, и словарём.
Полезно видеть оба решения рядом.

In [ ]:
import sqlite3

книги = [
    ("Мастер и Маргарита", "Булгаков"),
    ("Собачье сердце", "Булгаков"),
    ("Идиот", "Достоевский"),
    ("Му-му", "Тургенев"),
]

счёт = {}
for название, автор in книги:
    счёт[автор] = счёт.get(автор, 0) + 1
print("Python:", sorted(счёт.items()))

соединение = sqlite3.connect(":memory:")
курсор = соединение.cursor()
курсор.execute("CREATE TABLE книги (название TEXT, автор TEXT)")
курсор.executemany("INSERT INTO книги VALUES (?, ?)", книги)
соединение.commit()

результат = курсор.execute("""
    SELECT автор, COUNT(*) FROM книги GROUP BY автор ORDER BY автор
""").fetchall()
print("SQL:   ", результат)

На четырёх книгах разницы нет. На четырёх миллионах Python прочитает
всё в память, а база посчитает у себя и вернёт четыре строки —
и в этом вся суть: **вычисление идёт туда, где лежат данные**.

## Контрольная работа

Восемь задач по материалу четверти.

### Задача 1. Степени вершин

Функция получает список смежности и возвращает словарь
«вершина → степень».

In [ ]:
def степени(граф):
    return ...

In [ ]:
si.check("1", степени, [
    ({"А": ["Б", "В"], "Б": ["А"], "В": ["А"]}, {"А": 2, "Б": 1, "В": 1}),
    ({"А": []}, {"А": 0}),
])

### Задача 2. Обход в ширину

Функция возвращает список вершин в порядке обхода в ширину.

In [ ]:
def бфс(граф, старт):
    return ...

In [ ]:
si.check("2", бфс, [
    (({"А": ["Б", "В"], "Б": ["А", "Г"], "В": ["А"], "Г": ["Б"]}, "А"),
     ["А", "Б", "В", "Г"]),
    (({"А": ["Б"], "Б": ["А"]}, "Б"), ["Б", "А"]),
])

### Задача 3. Кратчайший путь

Функция получает взвешенный граф (словарь «вершина → {сосед: вес}»),
старт и финиш, возвращает длину кратчайшего пути. Если пути нет — `-1`.

In [ ]:
def кратчайший(граф, старт, финиш):
    return ...

In [ ]:
si.check("3", кратчайший, [
    (({"А": {"Б": 4, "В": 1}, "Б": {"А": 4, "Г": 2},
       "В": {"А": 1, "Г": 2}, "Г": {"Б": 2, "В": 2}}, "А", "Г"), 3),
    (({"А": {"Б": 5}, "Б": {"А": 5}, "В": {}}, "А", "В"), -1),
    (({"А": {"Б": 7}, "Б": {"А": 7}}, "А", "Б"), 7),
])

### Задача 4. Высота дерева

Функция возвращает высоту дерева; у дерева из одной вершины — 0.

In [ ]:
def высота(дерево, корень):
    return ...

In [ ]:
si.check("4", высота, [
    (({"А": ["Б", "В"], "Б": ["Г"], "В": [], "Г": []}, "А"), 2),
    (({"А": []}, "А"), 0),
])

### Задача 5. Количество путей

Функция считает число путей из старта в финиш в ориентированном
графе без циклов. Вершины обрабатывайте в алфавитном порядке.

In [ ]:
def путей(граф, старт, финиш):
    return ...

In [ ]:
si.check("5", путей, [
    (({"А": ["Б", "В"], "Б": ["Г"], "В": ["Г"], "Г": []}, "А", "Г"), 2),
    (({"А": ["Б"], "Б": ["В"], "В": []}, "А", "В"), 1),
])

### Готовим базу для задач 6–8

In [ ]:
курсор.execute("""CREATE TABLE сотрудники (
    id INTEGER PRIMARY KEY, фамилия TEXT, отдел INTEGER, зарплата INTEGER)""")
курсор.execute("""CREATE TABLE отделы (
    id INTEGER PRIMARY KEY, название TEXT)""")

курсор.executemany("INSERT INTO отделы VALUES (?, ?)", [
    (1, "разработка"), (2, "поддержка"), (3, "продажи"), (4, "архив"),
])
курсор.executemany("INSERT INTO сотрудники VALUES (?, ?, ?, ?)", [
    (1, "Иванов", 1, 120000),
    (2, "Петрова", 1, 140000),
    (3, "Сидоров", 1, 110000),
    (4, "Кузнецова", 2, 90000),
    (5, "Морозов", 2, 95000),
    (6, "Егорова", 3, 130000),
])
соединение.commit()
print("Готово")

### Задача 6. Средняя зарплата по отделам

Верните «название отдела, средняя зарплата (целое число)» только
для отделов, где есть сотрудники. По названию отдела.

In [ ]:
def средняя_по_отделам():
    return ...

In [ ]:
si.check("6", средняя_по_отделам, [
    ((), [("поддержка", 92500), ("продажи", 130000), ("разработка", 123333)]),
])

### Задача 7. Большие отделы

Верните названия отделов, где работает больше двух человек,
по алфавиту.

In [ ]:
def большие_отделы():
    return ...

In [ ]:
si.check("7", большие_отделы, [
    ((), [("разработка",)]),
])

### Задача 8. Отделы без сотрудников

Верните названия отделов, в которых нет ни одного сотрудника,
по алфавиту.

In [ ]:
def пустые_отделы():
    return ...

In [ ]:
si.check("8", пустые_отделы, [
    ((), [("архив",)]),
])

## Домашнее задание на каникулы

### Домашнее задание. Анализатор графа

Функция получает список смежности неориентированного графа
и возвращает список из четырёх значений:

1. количество вершин;
2. количество рёбер;
3. количество компонент связности;
4. `True`, если граф является деревом, иначе `False`.

Напомним: граф — дерево, если он связный и рёбер ровно на единицу
меньше, чем вершин.

In [ ]:
def анализ(граф):
    return ...

In [ ]:
si.check("дз", анализ, [
    ({"А": ["Б"], "Б": ["А"]}, [2, 1, 1, True]),
    ({"А": ["Б", "В"], "Б": ["А", "В"], "В": ["А", "Б"]}, [3, 3, 1, False]),
    ({"А": ["Б"], "Б": ["А"], "В": []}, [3, 1, 2, False]),
    ({"А": []}, [1, 0, 1, True]),
])

---

### Итоги четверти

Вы научились представлять связи графом и обходить его двумя
способами, находить кратчайшие пути алгоритмом Дейкстры, работать
с деревьями через рекурсию, проектировать реляционную базу данных
и спрашивать её на SQL — от простой выборки до группировок
и соединений.

В третьей четверти займёмся сетями, шифрованием и обработкой
табличных данных в Python, а дальше выйдем на прямую подготовку
к экзамену.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 17](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-17.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 19 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-19.ipynb)